# Contextual lattice operations

This notebook audits completed experiments; it does not silently run new
inference. Full vectors, selected components, and corpus-induced transport
are separate constructions. Category labels are not feature annotations.
See the frozen protocol in
`experiments/activation_studies/latticemethods_v1/PROTOCOL.md`.


In [1]:
from pathlib import Path
import json
import os
import sys

root = Path(os.environ.get("LATTICE_REPOSITORY", Path.cwd())).resolve()
while not (root / "src/lattmc").exists() and root != root.parent:
    root = root.parent
if not (root / "src/lattmc").exists():
    raise RuntimeError("Set LATTICE_REPOSITORY to the checkout root.")
sys.path[:0] = [str(root), str(root / "src")]
out = Path(os.environ.get(
    "LATTICE_METHOD_RESULTS",
    root / "data/activation_studies/latticemethods_v1"))
print("Interpreter:", sys.executable)
print("Records:", out)


Interpreter: /Users/ltsinadze/git/my_papers/.venv/bin/python
Records: /Users/ltsinadze/git/my_papers/data/activation_studies/latticemethods_v1


In [2]:
from lattmc.contextstudy.methodaudit_codexgen import audit

checks = audit(root, out)
print(checks)


{'dense_query_masks': 15840, 'token_gallery_rows': 29}


## Selected-component joins

P is category precision conditional on nonempty joined queries. Recall
includes empty queries. Delta P compares each nonempty joined query with
its stronger constituent before averaging. Repeated query--item pairs
are not independent examples; bootstrap intervals resample source groups.


In [3]:
import pandas as pd

summary = json.loads((out / "summary.json").read_text())
rows = []
for condition, values in summary.items():
    if "related" not in values:
        continue
    s = values["related"]
    rows.append({
        "condition": condition,
        "nonempty / 220": s["nonempty"],
        "precision (%)": 100 * s["precision"]["mean"],
        "recall (%)": 100 * s["recall"]["mean"],
        "delta P (pp)": 100 * s["paired_delta"]["mean"],
        "strict both (%)": 100 * s["strict"]["mean"],
    })
display(pd.DataFrame(rows).round(2))


,condition,nonempty / 220,precision (%),recall (%),delta P (pp),strict both (%)
0,ag_news/smol3,28,34.79,1.01,-6.86,12.73
1,ag_news/smol15,129,34.66,7.04,-5.21,57.73
2,ag_news/smol27,45,30.54,0.76,-10.64,39.09
3,ag_news/pythia_topk,208,26.22,29.05,-0.68,12.27
4,ag_news/qwen_transcoder,81,27.33,2.82,-7.03,45.45
5,ag_news/gemma_matryoshka,123,33.19,6.23,-4.61,49.09
6,dbpedia_14/smol3,57,28.02,2.49,-3.46,21.82
7,dbpedia_14/smol15,112,35.52,5.43,-0.54,57.27
8,dbpedia_14/smol27,69,27.57,1.79,-4.27,44.55
9,dbpedia_14/pythia_topk,220,6.02,36.97,-1.02,12.73


## Exact token witnesses

Rose means a token meets at least one positive requirement. Bold rose
means it meets the whole query. Yellow D means the pooled item satisfies
the query but no individual token does. N means a zero query. R means
rejection. These colors do not label semantic correctness. Text comes
from original cached token IDs, never the AI-edited CSV.


In [4]:
from IPython.display import HTML, display
from transformers import AutoTokenizer
from lattmc.contextstudy.witnessrender_codexgen import snippet

tokenizer = AutoTokenizer.from_pretrained("gpt2", local_files_only=True)
for kind in ("sae", "tc"):
    data = json.loads((out / f"gpt2_{kind}11.json").read_text())
    for row in data["gallery"]:
        if row.get("empty"):
            print(kind, row["query_name"], "no non-source member")
            continue
        row["valid_positions"] = list(range(128))
        row["diagnostic"] = (row["partial"] or [1])[0]
        status = row["status"]
        if status == "D":
            status = '<mark style="background:#FFEF9F">D</mark>'
        display(HTML(f'<b>{kind}, block 11, {row["query_name"]}; '
                     f'row {row["row"]}: {status}</b><p>'
                     + snippet(row, as_html=True) + '</p>'))
        print("Exact requirements:", row["query"])
        print("Complete original-token decode:", tokenizer.decode(
            row["tokens"], clean_up_tokenization_spaces=False))


Exact requirements: {'coordinates': [7275], 'values': [26.5772705078125]}
Complete original-token decode: <|endoftext|> shoes of a pauper.” [Plank/New Republic]

• Matthew Yglesias says John Edwards’s $400 haircut, over which there was such a huge kerfuffle, looks “like a rounding error compared to Palin’s September hair and makeup expenses.” It’s also surprising that this type of expenditure is legal, though apparently it is. [Think Progress]

• Chuck Todd and friends believe the story “could further add to the perception that Palin isn’t a serious candidate.” And Andrea Mitchell wonders whether Palin is �


Exact requirements: {'coordinates': [22858], 'values': [12.592910766601562]}
Complete original-token decode: <|endoftext|>ed investigation -- into the actress's links to drug lord Joaquin "El Chapo" Guzman and into the finances of her boutique tequila brand, Honor Del Castillo -- will determine if drug money went to her business.

There are indications that El Chapo provided illicit funds for del Castillo's tequila business, Gomez told the newspaper, but the links have not been proved.

"Even where there are indications, we need legal certainty," Gomez told El Universal.

Investigators are looking into communications between the actress, Guzman and his lawyer that included comments like "we're in business" and "


Exact requirements: {'coordinates': [], 'values': []}
Complete original-token decode: <|endoftext|> him from behind, Heat get offensive Rebound. This shouldn’t have been a big deal because this type of thing happens all the time. It occurred before the previous play shown, but I moved it to draw attention to what happened two minutes later. Noah gets called for his second foul for jumping behind LeBron James. Just after Bosh did the same. He sits for the remainder of the quarter. A fan provokes him, you know the rest. This wasn’t a foul. I have two replays of it. And if it was a foul, then how come Bosh is allowed to do the same thing?


Exact requirements: {'coordinates': [7275, 22858], 'values': [26.5772705078125, 12.592910766601562]}
Complete original-token decode: <|endoftext|> Other versions of the story substitute the viceroy of New Spain, such as Juan de Palafox y Mendoza in place of the archbishop.[5]

Selling mole mixes at the Feria Nacional del Mole in San Pedro Atocpan

Modern mole is a mixture of ingredients from North America, Europe and Africa, making it one of the first intercontinental dishes created in the Americas.[6] Its base, however, is indigenous. Nahuatl speakers had a preparation they called mōlli ( [ˈmoːlːi]), meaning "sauce", or ch


Exact requirements: {'coordinates': [7275, 22858], 'values': [26.5772705078125, 12.592910766601562]}
Complete original-token decode: <|endoftext|> him from behind, Heat get offensive Rebound. This shouldn’t have been a big deal because this type of thing happens all the time. It occurred before the previous play shown, but I moved it to draw attention to what happened two minutes later. Noah gets called for his second foul for jumping behind LeBron James. Just after Bosh did the same. He sits for the remainder of the quarter. A fan provokes him, you know the rest. This wasn’t a foul. I have two replays of it. And if it was a foul, then how come Bosh is allowed to do the same thing?


Exact requirements: {'coordinates': [9845], 'values': [7.741846084594727]}
Complete original-token decode: <|endoftext|> prudent, but the heat for this story will come from Republicans who cannot understand how their party would do something this stupid … during the collapse of the financial system and the probable beginning of a recession.” The wardrobe price tag is “without precedent,” and one can tell by the campaign’s “weakly defensive response” that they’re “deeply embarrassed.” [Atlantic]

• Steve Benen thinks the “political implications are more than a little humiliating” considering “all the McCain campaign messages a story like this steps on — ‘elitist


Exact requirements: {'coordinates': [212], 'values': [9.280964851379395]}
Complete original-token decode: <|endoftext|> years.

'We're all perplexed'

Alejandro Grattan, the editor in chief of the English-language El Ojo del Lago newspaper, told CBC News that he was a friend of the victims.

"We're all perplexed and we're in a daze," Grattan said in a telephone interview. "They had many, many friends down here."

He questioned the motive that police are giving.

"They both were beaten up savagely before they were killed, and you know, that's not exactly the kind of thing that somebody who's just there


Exact requirements: {'coordinates': [], 'values': []}
Complete original-token decode: <|endoftext|> him from behind, Heat get offensive Rebound. This shouldn’t have been a big deal because this type of thing happens all the time. It occurred before the previous play shown, but I moved it to draw attention to what happened two minutes later. Noah gets called for his second foul for jumping behind LeBron James. Just after Bosh did the same. He sits for the remainder of the quarter. A fan provokes him, you know the rest. This wasn’t a foul. I have two replays of it. And if it was a foul, then how come Bosh is allowed to do the same thing?
tc join no non-source member


Exact requirements: {'coordinates': [212, 9845], 'values': [9.280964851379395, 7.741846084594727]}
Complete original-token decode: <|endoftext|> him from behind, Heat get offensive Rebound. This shouldn’t have been a big deal because this type of thing happens all the time. It occurred before the previous play shown, but I moved it to draw attention to what happened two minutes later. Noah gets called for his second foul for jumping behind LeBron James. Just after Bosh did the same. He sits for the remainder of the quarter. A fan provokes him, you know the rest. This wasn’t a foul. I have two replays of it. And if it was a foul, then how come Bosh is allowed to do the same thing?


## Description transport

Training-set inclusion is guaranteed and checked. Held-out losses and
additional memberships are empirical results. They do not establish
neural propagation or monotonic semantic abstraction.


In [5]:
rows = []
for condition, values in summary.items():
    if "/transport/" in condition:
        rows.append({"condition": condition, **values})
display(pd.DataFrame(rows).round(4))
text_audit = json.loads((out / "corrected_text_audit.json").read_text())
print("AI-edited readings differing from raw decode:",
      text_audit["changed_from_raw_decode"], "/", text_audit["rows"])


,condition,eligible,before_precision,after_precision,added,correct_added,lost,zero
0,ag_news/transport/3-15,13,0.2933,0.2381,2428,597,15,0
1,ag_news/transport/15-27,18,0.2749,0.2490,2800,705,73,0
2,ag_news/transport/3-27,13,0.2933,0.2597,2656,676,10,0
3,dbpedia_14/transport/3-15,12,0.2344,0.0876,3024,215,27,0
4,dbpedia_14/transport/15-27,16,0.2103,0.0727,6161,409,28,0
5,dbpedia_14/transport/3-27,12,0.2344,0.0624,3172,213,28,0


AI-edited readings differing from raw decode: 823 / 25600


## Reproduction stages

Use the existing project uv environment and `PYTHONPATH=.:src`.
The CLI modules in `lattmc.contextstudy` accept explicit `--root`, `--out`,
and, where applicable, `--protocol` paths:

- `methodstudy_codexgen`: offline matrix evaluation.
- `tokenstudy_codexgen`: bounded GPT-2 source and witness replay.
- `methodgallery_codexgen`: original-batch MPS document replay.
- `methodtables_codexgen`: regenerated TeX presentation.
- `methodaudit_codexgen`: independent saved-array checks.

New semantic annotations and feature-mediated causal interventions have
not been executed. Full source-code meets do not transfer in this study;
selected joins and transported descriptions retain their failures.
